# Init

In [13]:
# init
from importlib import reload
import os
from pathlib import Path
import pandas as pd

import toolsGeneral.main as tgm
import toolsGeneral.logger as tgl
import toolsOSM.overpass as too
import toolsPandas.helpers as tph
import toolsSync.main as tsm

def pckgs_reload():
    reload(tgm)
    reload(too)
    reload(tph)
    reload(tgl)
    reload(tsm)

ROOT = Path.cwd().parents[0]
DATA_DIR = ROOT / "data"
TESTS_DIR = DATA_DIR / 'tests results'
CLEANED_DIR = DATA_DIR / 'cleaned'
TEST_BASIC_DIR = TESTS_DIR / 'osm basic test'
TEST_DUPLICATES_DIR = TESTS_DIR / 'osm duplicates test'
TEST_FIRST_LEVEL_DIR = TESTS_DIR / 'osm first level test'
FIXED_DIR = DATA_DIR / 'fixed'
FINAL_DIR = DATA_DIR / 'final'

logger = tgl.initiate_logger('logger', TEST_BASIC_DIR / 'basic_test.log')

cntr_meta = tgm.load(DATA_DIR / "osmMetaCountrDict.json")
cntr_id_to_name = {elem['id']:cntr for cntr, elem in cntr_meta.items()}

# Review

In [1]:
df_by_cntr_fixed = tgm.load_dirs(FIXED_DIR)
logger.info(len(df_by_cntr_fixed))

[2026-03-15 21:39:48] [INFO] : 85


In [3]:
df_all = pd.concat(df_by_cntr_fixed.values(), ignore_index=True)
print(len(df_all))

150266


In [4]:
df_all['tags.admin_level'].value_counts()

tags.admin_level
8    126278
6     22664
4      1241
2        83
Name: count, dtype: Int64

In [5]:
df_all['tags.parent_id'].map(lambda arg: 'Missing' if pd.isna(arg) else type(arg)).value_counts()

tags.parent_id
<class 'str'>    150183
Missing              83
Name: count, dtype: int64

In [7]:
def resume_divisions(rls : dict):
    resume = {}
    for k,df in rls.items():
        add_1 = df[df['tags.admin_level'] == '4']
        add_2 = df[df['tags.admin_level'] == '6']
        add_3 = df[df['tags.admin_level'] == '8']
        resume[k] = {'add_1': add_1, 'add_2': add_2, 'add_3': add_3}
    
    return pd.DataFrame(resume)

resume = resume_divisions(df_by_cntr_fixed)
len(resume)

3

## * all countries

In [ ]:
resume[list(df_by_cntr_fixed.keys())].T.map(len).peek()

,add_1,add_2,add_3
Afghanistan,34,0,0
Albania,3,12,374
Algeria,58,547,1541
Andorra,0,0,0
Angola,18,29,15
Anguilla,0,0,0
AntiguaAndBarbuda,3,8,0
Argentina,24,655,446
Armenia,11,15,0
Australia,15,600,0


## * one country

In [11]:
resume[['Colombia']].T.map(len).peek()

,add_1,add_2,add_3
Colombia,33,1128,2629


In [12]:
resume[['Colombia']].T.map(lambda arg: arg['tags.name'].to_list()).peek()

,add_1,add_2,add_3
Colombia,"[Caldas, Amazonas, Meta, Cundinamarca, Tolima, Antioquia, Atlántico, Bolívar, Cesar, Magdalena, Sucre, Córdoba, La Guajira, Chocó, Valle del Cauca, Norte de Santander, Quindío, Vichada, Vaupés, Santander, Risaralda, Putumayo, Nariño, Guaviare, Guainía, Arauca, Boyacá, Casanare, Cauca, Caquetá, Huila, Archipiélago de San Andrés, Providencia y Santa Catalina, Bogotá, Distrito Capital]","[Ulloa, Buenaventura, Florida, Cali, Yumbo, Pradera, Alcalá, Cartago, Obando, Andalucía, Ansermanuevo, Argelia, Bolívar, Buga, Bugalagrande, Zarzal, Caicedonia, Calima, El Cairo, El Águila, El Dovio, La Unión, Toro, Versalles, Roldanillo, La Victoria, Trujillo, San Pedro, Tuluá, Riofrío, Restrepo, Palmira, El Cerrito, Ginebra, Guacarí, Candelaria, Yotoco, Vijes, Dagua, Jamundí, Sevilla, La Cumbre, Puerto Santander, Puerto Nariño, Puerto Alegría, La Chorrera, Leticia, La Victoria, Tarapacá, Mirití-Paraná, Puerto Arica, El Encanto, La Pedrera, Putumayo, Lejanías, El Calvario, El Castillo, Guamal, Acacías, Restrepo, San Juanito, El Dorado, Mesetas, Uribe, Barranca de Upía, Cabuyaro, Cumaral, Puerto Gaitán, Mapiripán, Mapiripán, Puerto Concordia, La Macarena, Puerto Rico, Vista Hermosa, San Juan de Arama, Granada, Puerto López, Villavicencio, San Carlos de Guaroa, San Martín, Castilla La Nueva, Cubarral, Fuente de Oro, Puerto Lleras, Santa Rosalía, La Primavera, Puerto Carreño, Cumaribo, Valle del Guamuez (La Hormiga), San Miguel (La Dorada), Orito, Puerto Caicedo, Puerto Asís, Puerto Leguízamo, Mocoa, Santiago, Sibundoy, Colón, San Francisco, Puerto Guzmán, ...]","[Localidad Isla Cascajal, Localidad El Pailón, Putumayo, Teniente Manuel Clavero, Yaguas, Rosa Panduro, Comuna 3, Comuna 4, Comuna 5, Comuna 6, Comuna 9, Comuna 8, Comuna 2, Comuna 1, Comuna 7, Comuna 12, Comuna 11, Comuna 10, Vereda El Juncal, Vereda La Josefina, Vereda Ospina Pérez, Vereda Santa Rosa de Francisco, Vereda El Manzano, Vereda El Capulí, Vereda Iscuazán, Vereda Yaez, Vereda Santo Domingo, Vereda Las Delicias, Vereda Simón Bolívar, Vereda El Culantro, Vereda Aldea de María, Vereda Chorrera Negra, Vereda San Francisco, Vereda La Paz, Vereda El Contaderito, Vereda San Andrés, Vereda La Providencia, Vereda Las Cuevas, Vereda Santa Isabel, Vereda San José de Quisnamuez, Vereda Buenos Aires, Ciudad Yari, Vda. Camuya, Vda. Candilejas Este, Vda. Las Damas, Comuna Centro, Comuna Entreríos, Comuna Noroccidental, Comuna Oriente, Comuna Sur, Comuna La Floresta, Comuna Norte, Comuna Las Palmas, Comuna Suroriental, Comuna Nororiental, Vda. Mayoyoque, Vda Cristalina, Santa Rosa Baja, Vda. Gallinazo, Comuna Oriental, Comuna Suroriental, Comuna Norte, Comuna Occidental, San José de Caquetania, Vda. Las Delicias (El Morichal), Caño, Vda. El Jordán, El Triunfo, Vda. San Martín, El Retiro, Vda. Bocana del Perdido, Vda. California, Vda. Laureles, El Vergel, El Turpial, Vda. Palenque, Puerto Losada, Vda. El Palmar, Vda. El Rubí, Vda. La Cristalina, Vda. San Juan de Losada, Vereda La Eme, Vereda El Pepino, Vereda El Santuario, Vereda Alto Eslabón, Vereda La Florida, Vereda Las Mesas, Vereda La Tebaida, Vereda Los Andes, Vereda Las Planadas, Vereda Rumiyaco, Vereda El Diviso, Vereda Villa Nueva, San Luis De Chontoyaco, Vereda Las Palmeras 2, Vereda El Líbano, Vereda Villa Rosa, Vereda Peñas Blancas, Vereda Las Palmeras, Vereda San Carlos, ...]"


## * remove countries without first level

In [2]:
df_by_cntr_to_add = {}
for k,df in df_by_cntr_fixed.items():
    lvl1 = df[df['tags.admin_level'] == '4']
    if not lvl1.empty:
        df_by_cntr_to_add[k] = df
logger.info(len(df_by_cntr_fixed))
logger.info(len(df_by_cntr_to_add))

[2026-03-15 21:39:54] [INFO] : 85
[2026-03-15 21:39:54] [INFO] : 70


In [5]:
tgm.complement([k for k,v in df_by_cntr_fixed.items()],[k for k,v in df_by_cntr_to_add.items()])

['Andorra',
 'Anguilla',
 'Azerbaijan',
 'Bahamas',
 'Barbados',
 'Bermuda',
 'Estonia',
 'FaroeIslands',
 'Ireland',
 'IsleOfMan',
 'Latvia',
 'Montenegro',
 'NorthMacedonia',
 'SanMarino',
 'VaticanCity']

# Make add tree

# 1. rels_data_index

In [18]:
rels_index = {row['id']:row.to_dict() for df in df_by_cntr_to_add.values() for i,row in df.iterrows()}
len(rels_index)

147689

## 2. childs_index

In [19]:
childs_index = {}
for ele_id, vals in rels_index.items():
    parent_id = vals['tags.parent_id']
    if parent_id is not None:
        childs = childs_index.get(parent_id, [])
        childs.append(ele_id)
        childs_index[parent_id] = childs

## 3. make json tree by country and save

In [20]:
def make_tree_level(rels_ids):
    level = []
    for id in rels_ids:
        rel_data = rels_index[id]
        text_candidates = [
            rel_data.get('tags.name:en'),
            rel_data.get('tags.alt_name:en'), 
            rel_data.get('tags.name')]
        for text in text_candidates:
            if text:
                level.append({
                    'id': rel_data['id'],
                    'text': text,
                    'children': make_tree_level(childs_index.get(id, []))
                })
                break
    return level

In [21]:
lvl0_ids = []
for k,v in rels_index.items():
    if v['tags.admin_level'] == '2':
        lvl0_ids.append(k)

In [22]:
tree = make_tree_level(lvl0_ids)
print(len(tree))

68


In [23]:
for country_tree in tree:
    tgm.dump(FINAL_DIR / "add_tree" / f"{cntr_id_to_name[country_tree['id']]}.json", country_tree)

## 4. dump countries from original tree list, because I don't have the raw files

In [24]:
add_tree_missing = tgm.load(FINAL_DIR / "add_tree_missing.json")
print(len(add_tree_missing))

4


In [7]:
[elem['text'] for elem in add_tree_missing]

['Canada', 'Deutschland', 'France', 'Perú']

In [25]:
for country_tree in add_tree_missing:
    tgm.dump(FINAL_DIR / "add_tree" / f"{cntr_id_to_name[country_tree['id']]}.json", country_tree)

## 5. join add trees  

In [ ]:
all_add_tree = [tgm.load(f) for f in (FINAL_DIR / 'add_tree').rglob('*.json')]
print(len(all_add_tree))

70


In [37]:
tgm.dump(FINAL_DIR / "add_tree.json", all_add_tree)